# 1 · Light curves and I/O

Everything starts with a `LightCurve`. This notebook covers loading one, inspecting it, cutting it
down, converting between magnitudes and flux, and deriving phase and absolute magnitude.

**Needs:** the base install only.

In [1]:
from pathlib import Path

# Works whether the notebook is run from notebooks/ or from the repository root.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tests" / "data").is_dir())
AT2017GFO = ROOT / "tests" / "data" / "at2017gfo.csv"

import warnings
warnings.filterwarnings("ignore")           # keep the tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
import whisper_cbpf as wp

print("whisper_cbpf", wp.__version__)

whisper_cbpf 0.2.0


## Loading

`load_lightcurve` reads a CSV and works out what the columns mean. It accepts the usual synonyms
(`mjd`/`time`, `mag`/`magnitude`, `filter`/`band`, …), so most survey exports load unchanged.

In [2]:
lc = wp.load_lightcurve(AT2017GFO, redshift=0.0098)

print(f"{lc.name or 'AT2017GFO'}: {lc.n_points} points, data_mode={lc.data_mode!r}")
print("bands:", lc.bands)
print("redshift:", lc.redshift, "| known:", lc.redshift_known)

at2017gfo: 645 points, data_mode='magnitude'
bands: ['B', 'C', 'F110W', 'F160W', 'F336W', 'F475W', 'F606W', 'F625W', 'F775W', 'F814W', 'F850W', 'H', 'I', 'J', 'J1', 'K', 'Ks', 'R', 'U', 'V', 'W', 'Y', 'g', 'i', 'r', 'u', 'w', 'y', 'z']
redshift: 0.0098 | known: True


In [3]:
# A LightCurve IS an astropy Table, so everything you know about tables works.
lc[:5]

time,band,magnitude,magnitude_err,lambda_eff,zero_point,system
float64,str5,float64,float64,float64,float64,str7
57982.9814,i,17.48,0.02,7545.0,3631.0,AB
57982.9902,H,18.26,0.15,16620.0,3631.0,AB
57982.9919907,C,17.3335,0.06,nan,nan,unknown
57982.993,r,17.46,0.03,6215.0,3631.0,AB
57982.999,Ks,18.62,0.05,21590.0,3631.0,AB


## Selecting

`where()` takes `column=value` constraints with optional `_min` / `_max` / `_not` suffixes. On top
of it sit three convenience selectors.

In [4]:
print("g only                :", lc.select_bands(["g"]).n_points)
print("g or r                :", lc.select_bands(["g", "r"]).n_points)
print("SNR >= 5              :", lc.select_snr(min_snr=5.0).n_points)
print("where(band='g')       :", lc.where(band="g").n_points)
print("where(band_not='g')   :", lc.where(band_not="g").n_points)
print("where(time_min=57983) :", lc.where(time_min=57983.0).n_points)

g only                : 53
g or r                : 141
SNR >= 5              : 578
where(band='g')       : 53
where(band_not='g')   : 592
where(time_min=57983) : 640


In [5]:
# A typo names the columns that do exist, rather than raising a bare KeyError.
try:
    lc.where(magnitide=1.0)
except KeyError as exc:
    print(exc)

"where(): no column 'magnitide'; have ['time', 'band', 'magnitude', 'magnitude_err', 'lambda_eff', 'zero_point', 'system']"


## The one ordering rule that bites everyone

`select_time_window` filters whatever `time` currently holds. Call `set_explosion_date` **first**,
or you are comparing days-since-explosion against raw MJD and will quietly get an empty light curve.

In [6]:
right = lc.select_bands(["g"]).set_explosion_date(57982.529).select_time_window(0.0, 10.0)
wrong = lc.select_bands(["g"]).select_time_window(0.0, 10.0).set_explosion_date(57982.529)

print("set_explosion_date then window :", right.n_points, "points")
print("window then set_explosion_date :", wrong.n_points, "points  <- silently empty")

set_explosion_date then window : 53 points
window then set_explosion_date : 0 points  <- silently empty


## Magnitudes and flux

The table stores whichever it was given; `add_flux` and `add_mag` derive the other. Flux density is
in janskys, and the AB zero point is 3631 Jy.

In [7]:
both = lc.select_bands(["g"]).add_flux()
print("columns now:", [c for c in both.colnames if "flux" in c or "mag" in c])
print()
for row in both[:3]:
    print(f"  mag {row['magnitude']:.3f} +/- {row['magnitude_err']:.3f}"
          f"   ->  flux {row['flux']:.4e} Jy")

from whisper_cbpf.io.photometry import AB_ZEROPOINT_JY, flux_density_to_mag, mag_to_flux_density
print()
print("AB zero point:", AB_ZEROPOINT_JY, "Jy")
print("round trip   :", flux_density_to_mag(mag_to_flux_density(20.0)))

columns now: ['magnitude', 'magnitude_err', 'flux', 'flux_err']

  mag 17.410 +/- 0.020   ->  flux 3.9448e-04 Jy
  mag 17.410 +/- 0.040   ->  flux 3.9448e-04 Jy
  mag 17.390 +/- 0.020   ->  flux 4.0182e-04 Jy

AB zero point: 3631.0 Jy
round trip   : 20.0


## Derived quantities

`calc_phase` gives days since a reference (rest-frame if you pass a redshift); `calc_absmag`
converts to absolute magnitude via the distance modulus.

In [8]:
ph = lc.select_bands(["g"]).calc_phase(reference=57982.529, redshift=0.0098)
print("rest-frame phase, first 5 days:", np.round(ph["phase"][:5], 3))

am = lc.select_bands(["g"]).calc_absmag(redshift=0.0098)
print("absolute magnitude, first 5   :", np.round(am["absmag"][:5], 3))

rest-frame phase, first 5 days: [0.472 0.477 0.48  0.516 0.864]
absolute magnitude, first 5   : [-15.795 -15.795 -15.815 -15.885 -15.925]


## SNR, upper limits and export

In [9]:
print("SNR of the first 5 points:", np.round(lc.snr[:5], 1))
print("upper limits in this file:", int(np.sum(lc['upper_limit'])) if 'upper_limit' in lc.colnames else 0)

df = lc.to_dataframe()
print("\nto_dataframe ->", type(df).__name__, df.shape)
df.head(3)

SNR of the first 5 points: [54.3  7.2 18.1 36.2 21.7]
upper limits in this file: 0

to_dataframe -> DataFrame (645, 7)


,time,band,magnitude,magnitude_err,lambda_eff,zero_point,system
0,57982.981400,i,17.4800,0.02,7545.0,3631.0,AB
1,57982.990200,H,18.2600,0.15,16620.0,3631.0,AB
2,57982.991991,C,17.3335,0.06,NaN,NaN,unknown


## Alerts from ZTF and LSST

`survey=` reads a broker's or an alert packet's own fields and applies the survey's photometry
rule: difference-imaging PSF photometry (ZTF `magpsf`, LSST `psfFlux` in nJy), 5-sigma upper limits
from the non-detections, and the survey's band labels (`ztfg`, `lsstr`, ...). A row that is not a
measurement of the transient, such as a negative subtraction, is dropped and counted.

In [10]:
# An ALeRCE light-curve response: detections and non-detections, in ZTF's own field names.
alerce = {
    "detections": [
        {"mjd": 60001.2, "fid": 1, "magpsf": 19.20, "sigmapsf": 0.08, "isdiffpos": "t"},
        {"mjd": 60003.1, "fid": 2, "magpsf": 18.95, "sigmapsf": 0.07, "isdiffpos": "t"},
        {"mjd": 60005.2, "fid": 1, "magpsf": 18.90, "sigmapsf": 0.07, "isdiffpos": "t"},
        {"mjd": 60006.3, "fid": 2, "magpsf": 20.40, "sigmapsf": 0.20, "isdiffpos": "f"},  # negative
    ],
    "non_detections": [{"mjd": 59996.2, "fid": 1, "diffmaglim": 20.9},
                       {"mjd": 59999.1, "fid": 2, "diffmaglim": 20.7}],
}
ztf = wp.load_lightcurve(alerce, survey="ztf", name="ZTF example")

print(ztf.meta["photometry"])
print("dropped:", {k: v for k, v in ztf.meta["n_dropped"].items() if v})
# The last non-detection before the first detection bounds the explosion time.
print("explosion-time prior:", ztf.explosion_time_prior())
ztf.to_dataframe()[["time", "band", "magnitude", "magnitude_err", "upper_limit"]]

ZTF difference-image PSF magnitudes (magpsf/sigmapsf), positive subtractions (isdiffpos) only; magpsf_corr is not used
dropped: {'negative_flux': 1}
explosion-time prior: Uniform(59999.1, 60001.2)


,time,band,magnitude,magnitude_err,upper_limit
0,59996.2,ztfg,20.90,NaN,True
1,59999.1,ztfr,20.70,NaN,True
2,60001.2,ztfg,19.20,0.08,False
3,60003.1,ztfr,18.95,0.07,False
4,60005.2,ztfg,18.90,0.07,False


`survey="lsst"` reads Rubin alert packets the same way, and a model fit to either leaves the limits
before the first detection out of the likelihood: they only bound the explosion time. Notebook 11
takes an LSST alert from the packet to a ranked answer.

## Where next

* [02 · Bands, filters and SVO](02_bands_and_filters.ipynb) — how band names are resolved.
* [03 · Models](03_models.ipynb) — what you fit to this data.
* [11 · From an LSST alert to a ranked answer](11_lsst_alert_quickstart.ipynb) — the survey presets end to end.